In [1]:
# IMPORTANT: SOME KAGGLE DATA SOURCES ARE PRIVATE
# RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES.
import kagglehub
kagglehub.login()
#KGAT_945962f04ff95d8606b251f1d6c669aa
#KGAT_fcf3fbde8266d3f55f7b15d3b24258ba
#26a566de1711e46ce141d9d0339e633e
#prathapmanoharjoshi

/Users/prathap-4878/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


In [2]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        #print(os.path.join(dirname, filename))
        pass

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [3]:
# IMPORTANT: RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES,
# THEN FEEL FREE TO DELETE THIS CELL.
# NOTE: THIS NOTEBOOK ENVIRONMENT DIFFERS FROM KAGGLE'S PYTHON
# ENVIRONMENT SO THERE MAY BE MISSING LIBRARIES USED BY YOUR
# NOTEBOOK.

jan_2026_dl_gen_ai_project_path = kagglehub.competition_download('jan-2026-dl-gen-ai-project')

print('Data source import complete.')


Data source import complete.


In [4]:
jan_2026_dl_gen_ai_project_path

'/Users/prathap-4878/.cache/kagglehub/competitions/jan-2026-dl-gen-ai-project'

In [5]:
import os
import random
import numpy as np
import pandas as pd
import librosa
import torch
import torch.nn as nn
import wandb

from tqdm import tqdm
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score
from transformers import AutoFeatureExtractor, ASTForAudioClassification

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

wandb.login(key="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")

class CFG:
    SR = 22050
    AST_SR = 16000
    DURATION = 5
    AST_DURATION = 10
    SAMPLES = SR * DURATION
    AST_SAMPLES = AST_SR * AST_DURATION
    N_MELS = 64
    MEL_WIDTH = 216
    BATCH_SIZE = 16
    AST_BATCH = 8
    EPOCHS = 2
    AST_EPOCHS = 4
    LR = 1e-3
    AST_LR = 2e-5

#BASE_PATH = "/kaggle/input/competitions/jan-2026-dl-gen-ai-project/messy_mashup"
BASE_PATH = jan_2026_dl_gen_ai_project_path + "/messy_mashup"
TRAIN_PATH = f"{BASE_PATH}/genres_stems"
NOISE_PATH = f"{BASE_PATH}/ESC-50-master/audio"
TEST_PATH = f"{BASE_PATH}/mashups"
TEST_CSV = f"{BASE_PATH}/test.csv"

GENRES = ["blues","classical","country","disco","hiphop","jazz","metal","pop","reggae","rock"]

label2idx = {g:i for i,g in enumerate(GENRES)}
idx2label = {i:g for g,i in label2idx.items()}

data = []
for genre in GENRES:
    gpath = os.path.join(TRAIN_PATH, genre)
    for song in os.listdir(gpath):
        sp = os.path.join(gpath, song)
        if os.path.isdir(sp):
            data.append({"path": sp, "genre": genre})

df = pd.DataFrame(data)
train_df, val_df = train_test_split(df, test_size=0.2, stratify=df["genre"], random_state=42)

cpu


wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: Appending key for api.wandb.ai to your netrc file: /Users/prathap-4878/.netrc
wandb: Currently logged in as: 25ds3000125 (25ds3000125-iitmaana) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


In [6]:
train_df

,path,genre
575,/Users/prathap-4878/.cache/kagglehub/competiti...,jazz
204,/Users/prathap-4878/.cache/kagglehub/competiti...,country
964,/Users/prathap-4878/.cache/kagglehub/competiti...,rock
503,/Users/prathap-4878/.cache/kagglehub/competiti...,jazz
502,/Users/prathap-4878/.cache/kagglehub/competiti...,jazz
...,...,...
915,/Users/prathap-4878/.cache/kagglehub/competiti...,rock
346,/Users/prathap-4878/.cache/kagglehub/competiti...,disco
930,/Users/prathap-4878/.cache/kagglehub/competiti...,rock
306,/Users/prathap-4878/.cache/kagglehub/competiti...,disco


In [7]:
val_df

,path,genre
171,/Users/prathap-4878/.cache/kagglehub/competiti...,classical
92,/Users/prathap-4878/.cache/kagglehub/competiti...,blues
670,/Users/prathap-4878/.cache/kagglehub/competiti...,metal
535,/Users/prathap-4878/.cache/kagglehub/competiti...,jazz
21,/Users/prathap-4878/.cache/kagglehub/competiti...,blues
...,...,...
237,/Users/prathap-4878/.cache/kagglehub/competiti...,country
269,/Users/prathap-4878/.cache/kagglehub/competiti...,country
106,/Users/prathap-4878/.cache/kagglehub/competiti...,classical
983,/Users/prathap-4878/.cache/kagglehub/competiti...,rock


In [8]:
train_df = train_df.sample(100)
val_df = val_df.sample(50)

In [9]:
noise_files = []
for root,_,files in os.walk(NOISE_PATH):
    for f in files:
        if f.endswith(".wav"):
            noise_files.append(os.path.join(root,f))

def load_audio(path, sr, samples):
    y,_ = librosa.load(path, sr=sr)
    if len(y) < samples:
        y = np.pad(y,(0,samples-len(y)))
    return y[:samples]

def get_mel(y):
    mel = librosa.feature.melspectrogram(y=y, sr=CFG.SR, n_mels=CFG.N_MELS)
    mel = librosa.power_to_db(mel).astype(np.float32)
    if mel.shape[1] < CFG.MEL_WIDTH:
        mel = np.pad(mel, ((0,0),(0,CFG.MEL_WIDTH-mel.shape[1])))
    return mel[:,:CFG.MEL_WIDTH]

In [10]:
class CNNDataset(Dataset):
    def __init__(self, df, train=True):
        self.df = df.reset_index(drop=True)
        self.train = train
        self.stems = ["drums.wav","bass.wav","vocals.wav","others.wav"]

    def __len__(self):
        return len(self.df)

    def random_stem(self, genre):
        songs = self.df[self.df.genre==genre]["path"].tolist()
        for _ in range(10):
            song = random.choice(songs)
            stem = random.choice(self.stems)
            path = os.path.join(song, stem)
            if os.path.exists(path):
                return load_audio(path, CFG.SR, CFG.SAMPLES)
        return np.zeros(CFG.SAMPLES)

    def __getitem__(self, idx):
        genre = self.df.iloc[idx]["genre"]
        y = sum([self.random_stem(genre) for _ in range(4)]) / 4

        if self.train:
            try:
                y = librosa.effects.time_stretch(y, random.uniform(0.8,1.2))
            except:
                pass

        y = y[:CFG.SAMPLES] if len(y)>=CFG.SAMPLES else np.pad(y,(0,CFG.SAMPLES-len(y)))

        if self.train:
            noise = load_audio(random.choice(noise_files), CFG.SR, CFG.SAMPLES)
            y = y + random.uniform(0.1,0.5)*noise

        mel = np.expand_dims(get_mel(y),0)
        return torch.tensor(mel), torch.tensor(label2idx[genre])

class CNNModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(1,32,3,padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32,64,3,padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(64,128,3,padding=1), nn.ReLU(), nn.AdaptiveAvgPool2d(1)
        )
        self.fc = nn.Linear(128,10)

    def forward(self,x):
        x = self.conv(x)
        return self.fc(x.view(x.size(0),-1))

def macro_f1(y_true,y_pred):
    return f1_score(y_true,y_pred,average="macro")

In [11]:
def train_cnn():
    wandb.init(project="messy-mashup", name="cnn")
    model = CNNModel().to(device)
    opt = torch.optim.Adam(model.parameters(), lr=CFG.LR)
    crit = nn.CrossEntropyLoss()

    train_loader = DataLoader(CNNDataset(train_df), batch_size=CFG.BATCH_SIZE, shuffle=True)
    val_loader = DataLoader(CNNDataset(val_df,train=False), batch_size=CFG.BATCH_SIZE)

    for epoch in range(CFG.EPOCHS):
        model.train()
        tloss = 0
        for x,y in train_loader:
            x = x.to(device)
            y = y.to(device) 
            opt.zero_grad()
            out = model(x)
            loss = crit(out,y)
            loss.backward()
            opt.step()
            tloss += loss.item()

        model.eval()
        preds,targets = [],[]
        with torch.no_grad():
            for x,y in val_loader:
                x = x.to(device)
                out = model(x)
                preds.extend(torch.argmax(out,1).cpu().numpy())
                targets.extend(y.numpy())

        f1 = macro_f1(targets,preds)

        wandb.log({"epoch":epoch+1,"train_loss":tloss,"val_f1":f1})

    torch.save(model.state_dict(),"cnn.pth")

In [12]:
extractor = AutoFeatureExtractor.from_pretrained("MIT/ast-finetuned-audioset-10-10-0.4593")

def extract(audio):
    return extractor(audio, sampling_rate=CFG.AST_SR, return_tensors="pt")["input_values"].squeeze(0)

def create_mashup(paths):
    stems = ["drums.wav","vocals.wav","bass.wav","other.wav"]
    mix = np.zeros(CFG.AST_SAMPLES)
    for stem in stems:
        song = random.choice(paths)
        y = load_audio(os.path.join(song,stem), CFG.AST_SR, CFG.AST_SAMPLES)
        try:
            y = librosa.effects.time_stretch(y, random.uniform(0.8,1.2))
        except:
            pass
        mix += y
    noise = load_audio(random.choice(noise_files), CFG.AST_SR, CFG.AST_SAMPLES)
    mix += random.uniform(0.05,0.3)*noise
    return mix/(np.max(np.abs(mix))+1e-9)

def spec_aug(x):
    if random.random()<0.5:
        t = random.randint(0,100)
        t0 = random.randint(0,x.shape[0]-t)
        x[t0:t0+t,:]=0
    if random.random()<0.5:
        f = random.randint(0,20)
        f0 = random.randint(0,x.shape[1]-f)
        x[:,f0:f0+f]=0
    return x

class ASTDataset(Dataset):
    def __init__(self, df, train=True):
        self.df = df.reset_index(drop=True)
        self.train = train

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        genre = self.df.iloc[idx]["genre"]
        paths = self.df[self.df.genre==genre]["path"].tolist()
        audio = create_mashup(paths)
        feat = extract(audio)
        if self.train:
            feat = spec_aug(feat)
        return feat, GENRES.index(genre)

def mixup(x,y):
    lam = np.random.beta(0.4,0.4)
    idx = torch.randperm(x.size(0)).to(x.device)
    return lam*x+(1-lam)*x[idx], y, y[idx], lam


/Users/prathap-4878/Library/Python/3.9/lib/python/site-packages/transformers/audio_utils.py:525: UserWarning: At least one mel filter has all zero values. The value for `num_mel_filters` (128) may be set too high. Or, the value for `num_frequency_bins` (257) may be set too low.
  warnings.warn(


In [13]:
def train_ast(name, use_mixup=True):
    wandb.init(project="messy-mashup", name=name)

    model = ASTForAudioClassification.from_pretrained(
        "MIT/ast-finetuned-audioset-10-10-0.4593",
        num_labels=10,
        ignore_mismatched_sizes=True
    ).to(device)

    opt = torch.optim.AdamW(model.parameters(), lr=CFG.AST_LR)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=5)
    crit = nn.CrossEntropyLoss(label_smoothing=0.1)

    weights = train_df.genre.value_counts()
    sample_w = train_df.genre.map(lambda x:1.0/weights[x]).values
    sampler = WeightedRandomSampler(sample_w, len(sample_w))

    train_loader = DataLoader(ASTDataset(train_df), batch_size=CFG.AST_BATCH, sampler=sampler)
    val_loader = DataLoader(ASTDataset(val_df,False), batch_size=CFG.AST_BATCH)

    for epoch in range(CFG.AST_EPOCHS):
        model.train()
        tloss = 0

        for x,y in train_loader:
            x = x.to(device)
            y = y.to(device)
            if use_mixup and random.random()<0.5:
                x,y_a,y_b,lam = mixup(x,y)
                out = model(x).logits
                loss = lam*crit(out,y_a)+(1-lam)*crit(out,y_b)
            else:
                out = model(x).logits
                loss = crit(out,y)

            opt.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(),1.0)
            opt.step()

            tloss += loss.item()
            wandb.log({"step_loss":loss.item()})

        sched.step()

        model.eval()
        preds,targets=[],[]
        with torch.no_grad():
            for x,y in val_loader:
                x = x.to(device)
                out = model(x).logits
                preds.extend(torch.argmax(out,1).cpu().numpy())
                targets.extend(y.numpy())

        f1 = macro_f1(targets,preds)

        wandb.log({"epoch":epoch+1,"train_loss":tloss,"val_f1":f1})

    torch.save(model.state_dict(), f"{name}.pth")

In [14]:
train_cnn()
wandb.finish()

epoch,▁█
train_loss,█▁
val_f1,▁▁
epoch,2
train_loss,16.34939
val_f1,0.01132


In [15]:
train_ast("ast_v1", use_mixup=False)
wandb.finish()

Some weights of ASTForAudioClassification were not initialized from the model checkpoint at MIT/ast-finetuned-audioset-10-10-0.4593 and are newly initialized because the shapes did not match:
- classifier.dense.bias: found shape torch.Size([527]) in the checkpoint and torch.Size([10]) in the model instantiated
- classifier.dense.weight: found shape torch.Size([527, 768]) in the checkpoint and torch.Size([10, 768]) in the model instantiated
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


epoch,▁▃▆█
step_loss,██▇▇▇▅▃▆▄▄▅▄▃▃▄▃▂▃▃▃▂▃▂▂▄▂▂▂▂▂▂▂▂▁▁▂▁▂▁▁
train_loss,█▄▂▁
val_f1,▁▇▃█
epoch,4
step_loss,0.52772
train_loss,9.45642
val_f1,0.60564


In [16]:
train_ast("ast_v2", use_mixup=True)
wandb.finish()

Some weights of ASTForAudioClassification were not initialized from the model checkpoint at MIT/ast-finetuned-audioset-10-10-0.4593 and are newly initialized because the shapes did not match:
- classifier.dense.bias: found shape torch.Size([527]) in the checkpoint and torch.Size([10]) in the model instantiated
- classifier.dense.weight: found shape torch.Size([527, 768]) in the checkpoint and torch.Size([10, 768]) in the model instantiated
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


epoch,▁▃▆█
step_loss,██▆█▆▅▅▇▅▄▃▄▄▃▄▄▄▃▂▄▅▄▂▅▄▂▅▂▃▃▃▂▅▃▅▂▃▅▂▁
train_loss,█▃▁▁
val_f1,▁▄█▇
epoch,4
step_loss,0.62705
train_loss,14.98373
val_f1,0.6507


In [17]:
test_df = pd.read_csv(TEST_CSV)
test_df = test_df.head()
model = ASTForAudioClassification.from_pretrained(
    "MIT/ast-finetuned-audioset-10-10-0.4593",
    num_labels=10,
    ignore_mismatched_sizes=True
)

model.load_state_dict(torch.load("ast_v2.pth", map_location=device))
model.to(device)
model.eval()


def predict_tta(path, n=5):
    y = load_audio(path, CFG.AST_SR, CFG.AST_SAMPLES)
    y = y / (np.max(np.abs(y)) + 1e-9)

    preds = []

    for _ in range(n):
        start = random.randint(0, len(y) - CFG.AST_SAMPLES)
        crop = y[start:start + CFG.AST_SAMPLES]

        x = extract(crop).unsqueeze(0).to(device)

        with torch.no_grad():
            logits = model(x).logits
            probs = torch.softmax(logits, dim=1)
            preds.append(probs.cpu().numpy())

    preds = np.mean(preds, axis=0)
    return np.argmax(preds)


predictions = []

for _, row in tqdm(test_df.iterrows(), total=len(test_df)):
    if str(row["id"]).startswith("song"):
        file_path = f"{TEST_PATH}/{row['id']}.wav"
    else:
        file_path = f"{TEST_PATH}/song{int(row['id']):04d}.wav"

    pred = predict_tta(file_path, n=5)
    predictions.append(GENRES[pred])


submission = pd.DataFrame({
    "id": test_df["id"],
    "genre": predictions
})

submission.to_csv("submission.csv", index=False)

print("Submission saved successfully")

Some weights of ASTForAudioClassification were not initialized from the model checkpoint at MIT/ast-finetuned-audioset-10-10-0.4593 and are newly initialized because the shapes did not match:
- classifier.dense.bias: found shape torch.Size([527]) in the checkpoint and torch.Size([10]) in the model instantiated
- classifier.dense.weight: found shape torch.Size([527, 768]) in the checkpoint and torch.Size([10, 768]) in the model instantiated
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
100%|█████████████████████████████████████████████████████████████████████████████████████| 5/5 [00:03<00:00,  1.33it/s]

Submission saved successfully
